In [3]:
%load_ext autoreload
%autoreload 2
%reset -f

[autoreload of six failed: Traceback (most recent call last):
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 330, in update_class
    old_obj = getattr(old, key)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/six.py", line 98, in __get__
    setattr(obj, self.name, result)  # Invokes __set__.
AttributeError: 'NoneType' object has no attribute 'cStringIO'
]
[autoreload of dateutil.tz._factories failed: Traceback (most re

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of numpy failed: Traceback (most recent call last):
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 365, in update_class
    update_instances(old, new)
  File "/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 323, in update_instances
    object.__setattr__(ref, "__class__", new)
TypeError: __class__ assignment: 'errstate' object layout differs from 'errstate'
]
[autoreload of pan

# Imports

In [4]:
from locallib.picarrodb import *
from locallib.query import *
from locallib.box import *
from locallib.query import *

import pandas as pd
import geopandas as gpd
import contextily as ctx
import pandas as pd
import os

import sys
import os

from shapely import wkt
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from h3 import *

os.chdir('/home/sandbox/personal-repos/DA-3507/dump')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../lib')))
from custom_pandas import *

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully


/home/sandbox/personal-repos/.venv/lib/python3.9/site-packages/contextily/place.py:13: FutureWarning: In the future `np.bool` will be defined as the corresponding NumPy scalar.
  _val = np.random.randint(1000000)


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

## Get the surveys

In [ ]:
a = get_reports('Cadent',years = [2026]).execute([EU2_Conn])

In [ ]:
query = f"""SELECT SA.SurveyId as SurveyId, SA.Shape.STAsText() AS SurveyArea
FROM SurveyArea SA
WHERE SA.SurveyId IN (SELECT SurveyId FROM #TempSurvey)"""
#report_bc = a.iloc[[-1]].copy()
report_bc = a.copy()

In [ ]:
report_bc.db.set_query(query_SurveyH3Aggregation_byReport(report_table = '#TempReport'))
data =report_bc.db.execute([EU2_Conn], source_col = 'ReportId', temp_table_name = '#TempReport')
data.db.set_query(query)
surveys = data.db.execute([EU2_Conn], source_col = 'SurveyId', temp_table_name = '#TempSurvey')

In [ ]:
merged = pd.merge(data, surveys, on="SurveyId")
merged.BreadcrumbCounts.set_H3Resolution(11)
merged.BreadcrumbCounts.set_counts_only(False)
a = merged.BreadcrumbCounts.calculate('Breadcrumb','SurveyArea')
a['Passes'] = a['Intersections'] / 2

In [ ]:
a.to_parquet('BreadcrumbCounts.parquet')

In [ ]:
result = a.groupby("h3_cell").agg({"Passes": "sum", "SurveyId": "first"}).reset_index()
# INSERT_YOUR_CODE
import numpy as np

# Calculate the histogram for 'Passes' with edge bins between -0.5 and 0.5

# Define bins between -0.5 and 0.5, and cover full range of data
min_passes = result['Passes'].min()
max_passes = result['Passes'].max()
# Set the first bin edge <= min_passes, and last >= max_passes, and bin width = 1
start_edge = min(-0.5, np.floor(min_passes) - 0.5)
end_edge = max(0.5, np.ceil(max_passes) + 0.5)
bins = np.arange(start_edge, end_edge + 1, 1)

hist_counts, bin_edges = np.histogram(result['Passes'], bins=bins)
# Optionally, create a DataFrame for easier viewing
hist_df = pd.DataFrame({
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'bin_center': 0.5 * (bin_edges[:-1] + bin_edges[1:]),
    'count': hist_counts
})

hist_df.plot.bar(
    x='bin_center',
    y='count',
    legend=False,
    xlabel='Number of Passes (bin center)',
    ylabel='Count',
    title='Histogram of Passes'
)
plt.show()
hist_counts

In [ ]:
# INSERT_YOUR_CODE

import matplotlib.pyplot as plt

import contextily as ctx

# Plot the boundary
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the SurveyArea (boundary), should be a single row in 'boundary'
#boundary_gdf = boundary.copy()
#boundary_gdf['geometry'] = boundary_gdf['SurveyArea'].apply(lambda x: wkt.loads(x))
#boundary_gdf = gpd.GeoDataFrame(boundary_gdf, geometry='geometry', crs='EPSG:4326')
#boundary_gdf = boundary_gdf.to_crs(epsg=3857)
#boundary_gdf.plot(ax=ax, edgecolor='black', facecolor='none', linewidth=2, label='Boundary')

# Plot the Breadcrumbs
data_gdf = data.copy()
data_gdf['geometry'] = data_gdf['Breadcrumb'].apply(lambda x: wkt.loads(x))
data_gdf = gpd.GeoDataFrame(data_gdf, geometry='geometry', crs='EPSG:4326')
data_gdf = data_gdf.to_crs(epsg=3857)
data_gdf.plot(ax=ax, color='red', alpha=0.5, label='Breadcrumbs')

# INSERT_YOUR_CODE

# Plot the H3 hexagons with Intersections
from shapely.geometry import Polygon

def h3_cell_polygon(cell):
    # Returns a shapely Polygon for an h3 cell index
    boundary = h3.cell_to_boundary(cell)
    # The boundary comes as list of (lat, lng) tuples, so convert to (x, y) = (lng, lat)
    return Polygon([(lng, lat) for lat, lng in boundary])

# Create GeoDataFrame for hexagons with intersection counts
result_hex = result.copy()
result_hex['geometry'] = result_hex['h3_cell'].apply(h3_cell_polygon)
hex_gdf = gpd.GeoDataFrame(result_hex, geometry='geometry', crs="EPSG:4326")
hex_gdf = hex_gdf.to_crs(epsg=3857)

# Plot the hexagons with Intersections as the color (using a colormap)
hex_gdf.plot(
    ax=ax,
    column="Passes",
    alpha=0.6,
    edgecolor='k',
    linewidth=0.5,
    legend=True,
    label="H3 Hexagons"
)


# Add a base map using contextily
ctx.add_basemap(ax, source=ctx.providers.OpenTopoMap, crs="EPSG:3857")

ax.set_title("Boundary and Breadcrumbs")
ax.legend()
plt.show()